# 🦉 Strixlibe × Google Colab GPU LLM Server (Strata + Qwen3.8-Flash-Next 125B 超高知能版)
自律型ペンテストエージェント「Strixlibe」向けに、Google Colab（**NVIDIA L4 24GB GPU + ハイメモリ推奨**）上で、最新の階層オフロード推論エンジン **Strata** を使用して 1250億パラメータの超巨大 MoE モデル **Qwen3.8-Flash-Next** を爆速ホストする検証用ノートブックです。

### ⚡ Strata の特徴とメリット
- **125B MoE を L4 (24GB) で駆動**: VRAM・ホストRAM・CPUに階層分散配置することで、本来 A100 複数枚が必要な 125B モデルを単一 GPU で駆動。
- **50〜100+ tokens/sec の爆速レスポンス**: アクティブな Expert 層と N-gram テーブルの最適化により実用速度を実現。
- **256K 超長文コンテキスト & 思考チェーン**: ターゲットの大規模ソースコード監査や複雑な攻撃シナリオの立案に最適。
- **OpenAI 互換 API**: `https://<tunnel-url>/v1` で Strixlibe や Dispatcher からそのまま利用可能。

> **⚠️ ランタイム設定の注意**: Colab のメニュー「ランタイム」→「ランタイムのタイプを変更」で、**GPU（L4）** かつ **ハイメモリ（High-RAM: 約53GB）** を選択してください。

### 1. 設定項目（フォーム）

In [ ]:
#@title 🚀 サーバー基本設定 { run: "auto" }
MODEL_FAMILY = "qwen" #@param ["qwen", "coder"]
QUANT_LEVEL = "IQ2_XS" #@param ["IQ2_XS", "Q2_0", "IQ3_XXS", "IQ3_S"]
CONTEXT_LENGTH = 32768 #@param [16384, 32768, 65536, 131072]
HF_TOKEN = "" #@param {type:"string"}
DISPATCHER_URL = "https://strix-dispatcher-2acg5ejyka-uw.a.run.app" #@param {type:"string"}
DISPATCHER_API_KEY = "" #@param {type:"string"}
SLACK_WEBHOOK_URL = "" #@param {type:"string"}
IDLE_TIMEOUT_MINUTES = 25 #@param {type:"integer"}
TARGET_ENV = "all" #@param ["all", "staging", "production"]


### 2. 環境チェック & Strata / cloudflared のセットアップ（約1〜2分）

In [ ]:
import os
import psutil
import subprocess

# 1. GPU & システムスペック確認
!nvidia-smi

total_ram_gb = psutil.virtual_memory().total / (1024 ** 3)
print(f"\n[+] Total System RAM: {total_ram_gb:.1f} GB")
if total_ram_gb < 30:
    print("\n⚠️ [WARNING] システムRAMが30GB未満です。125BモデルのRAMオフロードでOOMが発生する恐れがあります。")
    print("⚠️ Colabのメニュー「ランタイム」→「ランタイムのタイプを変更」で「ハイメモリ」を選択してください。\n")

# 2. 残存プロセスのクリーンアップ
!pkill -9 -f strata || true
!pkill -9 -f cloudflared || true

# 3. システム依存パッケージの完全導入（python3-venvの競合防止）
!apt-get update -qq && apt-get install -y -qq git zstd curl pciutils python3-venv python3.12-venv python3-pip
!pip install -q --no-warn-conflicts hf-transfer
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    !huggingface-cli login --token $HF_TOKEN --add-to-git-credential || true

# 4. cloudflared の導入
!curl -s -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# 5. Strata リポジトリのクローンと仮想環境の事前作成
if not os.path.exists("/content/Strata"):
    print("[*] Cloning Strata repository...")
    !git clone https://github.com/Niko1221/Strata.git /content/Strata
else:
    print("[*] Updating Strata repository...")
    !cd /content/Strata && git pull

!chmod +x /content/Strata/*.sh || true

# Colab専用: .venv を事前に確実に構築
if not os.path.exists("/content/Strata/.venv/bin/python"):
    print("[*] Creating dedicated virtualenv in Strata...")
    !python3 -m venv /content/Strata/.venv
    !/content/Strata/.venv/bin/pip install -q --upgrade pip setuptools wheel

print("\n[+] Strata environment is fully prepared!")


### 3. Qwen3.8-Flash-Next モデルの準備（約5〜8分）

In [ ]:
# setup.sh のOSシェル判定をバイパスし、ColabのPythonで直接setup.pyを実行
print(f"[*] Configuring & Downloading Strata model: {MODEL_FAMILY} ({QUANT_LEVEL}, ctx: {CONTEXT_LENGTH}) ...")
print("[*] Downloading model weights & building CUDA cache (approx 3-7 mins) ...\n")

!cd /content/Strata && python3 setup.py --yes --family {MODEL_FAMILY} --model {QUANT_LEVEL} --context {CONTEXT_LENGTH} --host 0.0.0.0 --port 8080 --no-start

print("\n[+] Model weights downloaded and cached successfully!")


### 4. Strata サーバー & Cloudflare Tunnel の起動

In [ ]:
import os
import glob
import subprocess
import time
import re
import requests

strata_dir = "/content/Strata"
print("[*] Starting Strata server in background...")

# ColabのPythonで直接setup.pyサーバーを起動
launch_cmd = ["python3", f"{strata_dir}/setup.py", "--family", MODEL_FAMILY, "--model", QUANT_LEVEL, "--context", str(CONTEXT_LENGTH), "--host", "0.0.0.0", "--port", "8080", "--yes"]

strata_log = open("/content/strata.log", "w")
server_proc = subprocess.Popen(launch_cmd, cwd=strata_dir, stdout=strata_log, stderr=subprocess.STDOUT)

# サーバー起動待機 (ヘルスチェック)
print("[*] Waiting for Strata server to initialize weights into GPU & RAM (approx 1-2 mins) ...")
ready = False
for i in range(90):
    try:
        r = requests.get("http://127.0.0.1:8080/health", timeout=1)
        if r.status_code == 200:
            ready = True
            break
    except:
        try:
            r = requests.get("http://127.0.0.1:8080/v1/models", timeout=1)
            if r.status_code == 200:
                ready = True
                break
        except:
            pass
    time.sleep(2)

if not ready:
    print("[-] Health check still pending. Last 20 lines of /content/strata.log:")
    !tail -n 20 /content/strata.log || true
else:
    print("[+] Strata server is UP and healthy on port 8080!")

# Cloudflare Tunnel の起動
print("[*] Establishing Cloudflare Tunnel...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--protocol", "http2", "--url", "http://localhost:8080", "--http-host-header", "localhost:8080"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

tunnel_url = None
while True:
    line = tunnel_proc.stdout.readline()
    if not line:
        break
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        print(f"\n🎉 ==========================================================")
        print(f"[+] Cloudflare Tunnel established successfully!")
        print(f"[+] Public URL: {tunnel_url}")
        print(f"[+] OpenAI API Base: {tunnel_url}/v1")
        print(f"==========================================================\n")
        break

api_base = f"{tunnel_url}/v1"
ACTIVE_MODEL_NAME = f"qwen3.8-flash-next-{QUANT_LEVEL.lower()}"


### 5. 疎通テスト（125B 推論ベンチマーク）

In [ ]:
print(f"[*] Sending test prompt to: {api_base}/chat/completions ...")

test_payload = {
    "model": ACTIVE_MODEL_NAME,
    "messages": [
        {"role": "user", "content": "Strixペンテストエージェントの125B推論基盤テストです。自律ペンテストにおいてあなたの得意とする役割を日本語で簡潔に2文以内で述べてください。"}
    ],
    "max_tokens": 200,
    "temperature": 0.6
}

try:
    resp = requests.post(f"{api_base}/chat/completions", json=test_payload, timeout=120)
    if resp.status_code == 200:
        result = resp.json()
        reply = result["choices"][0]["message"]["content"]
        print("\n[+] 125B 推論テスト大成功！モデルからの返答:")
        print("--------------------------------------------------")
        print(reply.strip())
        print("--------------------------------------------------")
    else:
        print(f"[-] テスト失敗: HTTP {resp.status_code} - {resp.text}")
except Exception as e:
    print(f"[-] テスト例外: {e}")


### 6. Dispatcher 自動同期 & Slack 通知

In [ ]:
# Dispatcher へのセッション登録
if DISPATCHER_URL and DISPATCHER_API_KEY:
    try:
        reg_payload = {
            "provider": "colab-strata",
            "base_url": api_base,
            "model_name": ACTIVE_MODEL_NAME,
            "env_target": TARGET_ENV,
            "ttl_minutes": 120
        }
        headers = {"X-API-Key": DISPATCHER_API_KEY, "Content-Type": "application/json"}
        reg_res = requests.post(f"{DISPATCHER_URL}/api/v1/llm-session", json=reg_payload, headers=headers, timeout=10)
        print(f"[+] Registered to Dispatcher: status {reg_res.status_code}")
    except Exception as e:
        print(f"[-] Dispatcher registration skipped or failed: {e}")
else:
    print("[*] Dispatcher 未設定のため同期をスキップしました（手動接続可能モード）。")

# Slack 通知
if SLACK_WEBHOOK_URL:
    try:
        msg = (
            f"🧠 *Strixlibe Colab Strata 125B Server is ONLINE!*\n"
            f"• *Engine*: Strata (Hierarchical Offload)\n"
            f"• *Model*: `Qwen3.8-Flash-Next ({QUANT_LEVEL})`\n"
            f"• *Endpoint*: `{api_base}`\n"
            f"• *Context*: {CONTEXT_LENGTH} tokens\n"
            f"• *Watchdog*: {IDLE_TIMEOUT_MINUTES} mins auto-kill\n"
            f"💡 Strixlibe will route autonomous scans to this ultra-high-intellect model."
        )
        requests.post(SLACK_WEBHOOK_URL, json={"text": msg}, timeout=5)
        print("[+] Slack notification sent.")
    except Exception as e:
        print(f"[-] Slack notification failed: {e}")


### 7. CU保護ウォッチドッグ（バックグラウンド監視・自動シャットダウン）

In [ ]:
import threading
from google.colab import runtime

def watchdog_worker():
    print(f"[*] Watchdog background worker started. Auto-terminate after {IDLE_TIMEOUT_MINUTES}m inactivity.")
    last_seen_time = time.time()
    idle_timeout_seconds = IDLE_TIMEOUT_MINUTES * 60
    
    while True:
        time.sleep(30)
        elapsed = time.time() - last_seen_time
        if elapsed > idle_timeout_seconds:
            print(f"[!] Inactivity threshold exceeded. Terminating runtime to preserve Compute Units.")
            if SLACK_WEBHOOK_URL:
                try:
                    requests.post(SLACK_WEBHOOK_URL, json={"text": "🛑 *Colab Strata 125B Server disconnected* (Idle timeout)."}, timeout=5)
                except:
                    pass
            runtime.unassign()
            break

t = threading.Thread(target=watchdog_worker, daemon=True)
t.start()
print("[OK] All systems operational! Strata 125B is running and watchdog is active.")
